# Evaluation: WER, CER, and Normalization

Before you can improve a speech recognition system, you need to measure it honestly.
This notebook covers the two standard metrics for ASR evaluation, text normalization,
and how to report results across multiple languages.

In [ ]:
# !pip install jiwer
import re
import string
from collections import defaultdict
import jiwer

## Word Error Rate

WER compares a model's transcription (the hypothesis) against a human-written transcript (the reference).
It counts three types of mistakes at the word level:

- **Substitution**: the wrong word was produced (`cat` instead of `bat`)
- **Deletion**: a word was dropped entirely
- **Insertion**: an extra word was added that shouldn't be there

```
WER = (Substitutions + Deletions + Insertions) / len(reference_words)
```

A WER of 0.0 means perfect transcription. A WER of 1.0 means every reference word has an error.
It's possible to exceed 1.0 if there are many insertions.

The editing operations are computed using dynamic programming, the same algorithm behind
spell-checkers and diff tools.

### WER Formula and Worked Numeric Example

```
WER = (S + D + I) / N
```

Where:
- S = number of substitutions
- D = number of deletions
- I = number of insertions
- N = number of words in the reference

Walk through a concrete example step by step.

In [ ]:
# WER = (S + D + I) / N
# Worked numeric example:
#
# Reference:   "the    quick  brown  fox   jumps  over   the  lazy  dog"
# Hypothesis:  "the    quick  brown  cat   jumped        the  lazy  dogs"
#
# Alignment:
#   the   -> the    (correct)
#   quick -> quick  (correct)
#   brown -> brown  (correct)
#   fox   -> cat    (SUBSTITUTION: "cat" for "fox")
#   jumps -> jumped (SUBSTITUTION: "jumped" for "jumps")
#   over  -> (none) (DELETION: "over" was dropped)
#   the   -> the    (correct)
#   lazy  -> lazy   (correct)
#   dog   -> dogs   (SUBSTITUTION: "dogs" for "dog")
#
# S=3, D=1, I=0, N=9

S = 3   # substitutions
D = 1   # deletions
I = 0   # insertions
N = 9   # reference word count

WER = (S + D + I) / N

print("Reference : the quick brown fox jumps over the lazy dog")
print("Hypothesis: the quick brown cat jumped       the lazy dogs")
print()
print(f"  Substitutions (S) : {S}  (fox->cat, jumps->jumped, dog->dogs)")
print(f"  Deletions     (D) : {D}  ('over' was dropped)")
print(f"  Insertions    (I) : {I}")
print(f"  Reference len (N) : {N}")
print()
print(f"  WER = ({S} + {D} + {I}) / {N} = {S+D+I}/{N} = {WER:.4f}  ({WER*100:.1f}%)")

# Verify with jiwer
import jiwer
ref = "the quick brown fox jumps over the lazy dog"
hyp = "the quick brown cat jumped the lazy dogs"
jiwer_wer = jiwer.wer(ref, hyp)
measures = jiwer.compute_measures(ref, hyp)
print()
print(f"  jiwer verification:")
print(f"    WER              : {jiwer_wer:.4f}")
print(f"    Substitutions    : {measures['substitutions']}")
print(f"    Deletions        : {measures['deletions']}")
print(f"    Insertions       : {measures['insertions']}")

In [ ]:
def edit_distance(ref_tokens, hyp_tokens):
    """Compute the minimum edit distance (Levenshtein) between two token lists.
    Returns (distance, substitutions, deletions, insertions).
    """
    n, m = len(ref_tokens), len(hyp_tokens)
    # dp[i][j] = (total_edits, subs, dels, ins)
    dp = [[(0, 0, 0, 0)] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = (i, 0, i, 0)  # i deletions
    for j in range(1, m + 1):
        dp[0][j] = (j, 0, 0, j)  # j insertions

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # no edit
            else:
                sub = dp[i - 1][j - 1]
                dele = dp[i - 1][j]
                ins = dp[i][j - 1]
                best = min(sub[0] + 1, dele[0] + 1, ins[0] + 1)
                if best == sub[0] + 1:
                    dp[i][j] = (best, sub[1] + 1, sub[2], sub[3])
                elif best == dele[0] + 1:
                    dp[i][j] = (best, dele[1], dele[2] + 1, dele[3])
                else:
                    dp[i][j] = (best, ins[1], ins[2], ins[3] + 1)

    return dp[n][m]


def wer(reference: str, hypothesis: str) -> float:
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    if len(ref_tokens) == 0:
        return 0.0 if len(hyp_tokens) == 0 else 1.0
    edits, subs, dels, ins = edit_distance(ref_tokens, hyp_tokens)
    return edits / len(ref_tokens)


# Quick sanity check
ref = "the cat sat on the mat"
hyp = "the cat sat on a mat"  # one substitution: 'a' for 'the'
score = wer(ref, hyp)
print(f"WER (from scratch): {score:.4f}")
print(f"WER (jiwer):        {jiwer.wer(ref, hyp):.4f}")

## Character Error Rate

CER applies the same edit-distance logic, but at the character level instead of the word level.
It's more useful when:

- The language doesn't use spaces to separate words (Chinese, Japanese, Thai)
- You're working with morphologically rich languages where one word mistake inflates WER unfairly
- You want finer-grained signal for closely related transcription errors

```
CER = (char substitutions + char deletions + char insertions) / len(reference_chars)
```

### CER Formula and Character-Level Alignment Example

```
CER = (char_S + char_D + char_I) / len(reference_characters)
```

The key difference from WER: spaces are removed before counting, and each individual character is a unit. A single word error that touches 5 characters will contribute 5 units of edit distance to CER but only 1 to WER.

In [ ]:
# CER character-level alignment example
#
# Reference  : "recognition"   (11 chars, spaces removed)
# Hypothesis : "recgnition"    (10 chars -- missing 'o')
#
# Optimal alignment (one deletion at position 4):
#   r e c o g n i t i o n
#   r e c - g n i t i o n
#       |   ^
#       |   deletion of 'o'
#
# char_S=0, char_D=1, char_I=0, N=11
# CER = 1/11 = 0.0909

ref_str = "recognition"
hyp_str = "recgnition"

ref_chars = list(ref_str)
hyp_chars = list(hyp_str)

print(f"Reference  : '{ref_str}'  ({len(ref_chars)} chars)")
print(f"Hypothesis : '{hyp_str}'  ({len(hyp_chars)} chars)")
print()

# Visual character-level alignment
print("Alignment (| = match, D = deletion):")
alignment_ref = []
alignment_hyp = []
alignment_ops = []

# We know the optimal path for this example manually:
# r-r, e-e, c-c, o-DEL, g-g, n-n, i-i, t-t, i-i, o-o, n-n
manual_ref_aligned = list("r e c o g n i t i o n")
manual_hyp_aligned = list("r e c   g n i t i o n")  # space = deletion
print("  REF: " + " ".join(c if c != " " else "-" for c in "recognition"))
print("  HYP: " + " ".join(c if c else "-" for c in list("rec") + [" "] + list("gnition")))
print("  OPS: " + "= = = D = = = = = = =")
print()

# Use jiwer for accurate CER
jiwer_cer = jiwer.cer(ref_str, hyp_str)
print(f"CER = 1 deletion / 11 reference chars = {1/11:.4f}")
print(f"jiwer.cer verification: {jiwer_cer:.4f}")

# Show that WER is much harsher for this case
jiwer_wer = jiwer.wer(ref_str, hyp_str)
print()
print(f"Word-level comparison:")
print(f"  WER = {jiwer_wer:.4f}  (1 wrong word out of 1 = 100% error)")
print(f"  CER = {jiwer_cer:.4f}  (1 wrong char out of 11 = 9% error)")
print()
print("CER is more informative here: only one character was dropped.")

In [ ]:
def cer(reference: str, hypothesis: str) -> float:
    ref_chars = list(reference.replace(" ", ""))
    hyp_chars = list(hypothesis.replace(" ", ""))
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    edits, _, _, _ = edit_distance(ref_chars, hyp_chars)
    return edits / len(ref_chars)


pairs = [
    ("hello world", "hello world"),       # perfect
    ("hello world", "hello wrold"),        # one transposition (counts as 2 edits at char level)
    ("the quick brown fox", "the quikc brown fox"),  # typo
    ("recognition", "recgnition"),         # deletion
]

print(f"{'Reference':<25} {'Hypothesis':<25} {'WER':>6} {'CER':>6}")
print("-" * 68)
for ref, hyp in pairs:
    print(f"{ref:<25} {hyp:<25} {wer(ref, hyp):>6.3f} {cer(ref, hyp):>6.3f}")

## Why Normalization Matters

Consider this reference transcript: `"The flight costs $249."` 
A model might produce: `"the flight costs two hundred forty nine dollars"`

That's a semantically identical transcription, but raw WER will penalize it heavily.
Text normalization converts both reference and hypothesis to a common form before scoring,
making the evaluation reflect actual understanding rather than surface formatting.

Common normalization steps:
- Lowercase everything
- Remove punctuation
- Expand or collapse number forms
- Strip extra whitespace

Normalization is a judgment call. More aggressive normalization raises scores;
the key is applying the same normalization to both reference and hypothesis,
and documenting what you did when reporting results.

In [ ]:
def normalize_text(text: str) -> str:
    """Basic normalization: lowercase, strip punctuation, collapse whitespace."""
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Show how normalization affects scores
pairs_norm = [
    (
        "Hello, World! How are you?",
        "hello world how are you",
    ),
    (
        "The meeting is at 3 P.M.",
        "the meeting is at 3 pm",
    ),
    (
        "Dr. Smith said: 'It's fine.'",
        "dr smith said its fine",
    ),
]

print(f"{'Reference':<35} {'Hypothesis':<35} {'Raw WER':>8} {'Norm WER':>9}")
print("-" * 92)
for ref, hyp in pairs_norm:
    raw = wer(ref, hyp)
    norm = wer(normalize_text(ref), normalize_text(hyp))
    print(f"{ref:<35} {hyp:<35} {raw:>8.3f} {norm:>9.3f}")

## Per-Language Reporting

A multilingual ASR system will perform differently across languages.
Reporting only an aggregate WER hides these gaps. A system with 5% WER on English
and 40% WER on Swahili would report a deceivingly low average if English dominates
the evaluation set.

Always report per-language metrics alongside the aggregate, and note the number
of test examples per language so readers can judge reliability.

In [ ]:
def evaluate_by_language(data: dict) -> None:
    """
    data: {lang_code: [(reference, hypothesis), ...]}
    Prints a summary table of WER and CER per language.
    """
    print(f"{'Language':<12} {'Examples':>8} {'WER':>8} {'CER':>8}")
    print("-" * 42)

    all_wers, all_cers = [], []
    for lang, pairs in sorted(data.items()):
        lang_wers = [wer(normalize_text(r), normalize_text(h)) for r, h in pairs]
        lang_cers = [cer(normalize_text(r), normalize_text(h)) for r, h in pairs]
        mean_wer = sum(lang_wers) / len(lang_wers)
        mean_cer = sum(lang_cers) / len(lang_cers)
        all_wers.extend(lang_wers)
        all_cers.extend(lang_cers)
        print(f"{lang:<12} {len(pairs):>8} {mean_wer:>8.3f} {mean_cer:>8.3f}")

    print("-" * 42)
    print(
        f"{'Overall':<12} {len(all_wers):>8} "
        f"{sum(all_wers)/len(all_wers):>8.3f} "
        f"{sum(all_cers)/len(all_cers):>8.3f}"
    )

## Hands-On: Evaluate a Multilingual Pipeline

The synthetic data below represents hypotheses from an ASR model across three languages.
Run the evaluation, identify which language has the worst performance, and explain why
the normalization step matters for the English examples.

In [ ]:
# Synthetic ASR results: (reference, hypothesis) per language
eval_data = {
    "en": [
        ("The train arrives at half past three.", "the train arrives at half past three"),
        ("She ordered two coffees and a sandwich.", "she ordered two coffees and a sandwitch"),
        ("Call Dr. Johnson immediately.", "call dr johnson immediately"),
        ("It costs $45.99.", "it costs forty five dollars and ninety nine cents"),
        ("I'll see you on Monday.", "ill see you on monday"),
    ],
    "fr": [
        ("Bonjour, comment allez-vous?", "bonjour comment allez vous"),
        ("Je voudrais un cafe, s'il vous plait.", "je voudrais un cafe sil vous plait"),
        ("La reunion commence a quatorze heures.", "la reunion commence a quatorze heures"),
        ("Ou est la gare?", "ou est la gare"),
        ("Merci beaucoup pour votre aide.", "merci beaucoup pour votre aides"),
    ],
    "sw": [
        ("Habari yako?", "habari yako"),
        ("Ninataka kwenda sokoni.", "ninataka kwenda sokoin"),
        ("Yeye ni mwalimu mzuri.", "yee ni mwalimu mzuri"),
        ("Tunakwenda shuleni kesho.", "tunakwenda shuleini kesho"),
        ("Asante sana kwa msaada wako.", "asante sana kwa msaada wake"),
    ],
}

In [ ]:
print("=== Normalized Evaluation ===")
evaluate_by_language(eval_data)

print()
print("=== Raw (no normalization) ===")

def evaluate_raw(data):
    print(f"{'Language':<12} {'Examples':>8} {'WER':>8} {'CER':>8}")
    print("-" * 42)
    for lang, pairs in sorted(data.items()):
        lang_wers = [wer(r, h) for r, h in pairs]
        lang_cers = [cer(r, h) for r, h in pairs]
        print(
            f"{lang:<12} {len(pairs):>8} "
            f"{sum(lang_wers)/len(lang_wers):>8.3f} "
            f"{sum(lang_cers)/len(lang_cers):>8.3f}"
        )

evaluate_raw(eval_data)

In [ ]:
# Exercise questions (answer by modifying this cell):
#
# 1. Which language has the highest WER after normalization?
#    Your answer:
#
# 2. For English, find the example where normalization changes the WER the most.
#    Print the raw WER and normalized WER for each English pair.

print("English pair analysis:")
print(f"{'Reference':<50} {'Hypothesis':<50} {'Raw':>6} {'Norm':>6}")
for ref, hyp in eval_data["en"]:
    r = wer(ref, hyp)
    n = wer(normalize_text(ref), normalize_text(hyp))
    print(f"{ref:<50} {hyp:<50} {r:>6.3f} {n:>6.3f}")

## Using jiwer for Production Evaluation

The functions above are useful for understanding the mechanics, but `jiwer` is faster
and handles edge cases more robustly for production use.
It also provides a `Compose` transform pipeline for normalization.

In [ ]:
import jiwer

transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])

ref = "The train arrives at half past three."
hyp = "the train arrives at half past three"

score = jiwer.wer(
    ref, hyp,
    reference_transform=transformation,
    hypothesis_transform=transformation,
)
print(f"jiwer WER with normalization pipeline: {score:.4f}")

## Full jiwer Usage: compute_measures, CER, and Alignment Visualization

`jiwer.compute_measures` returns the full breakdown (S, D, I, hits) in addition to WER. The `visualize_alignment` function prints a readable diff between reference and hypothesis words.

In [ ]:
import jiwer

# --- compute_measures: full breakdown ---
reference  = "the quick brown fox jumps over the lazy dog"
hypothesis = "the quick brown cat jumped the lazy dogs"

measures = jiwer.compute_measures(reference, hypothesis)
print("=== compute_measures output ===")
print(f"  WER              : {measures['wer']:.4f}  ({measures['wer']*100:.1f}%)")
print(f"  MER              : {measures['mer']:.4f}  (Match Error Rate)")
print(f"  WIL              : {measures['wil']:.4f}  (Word Info Lost)")
print(f"  Hits             : {measures['hits']}")
print(f"  Substitutions    : {measures['substitutions']}")
print(f"  Deletions        : {measures['deletions']}")
print(f"  Insertions       : {measures['insertions']}")

# --- CER ---
cer_score = jiwer.cer(reference, hypothesis)
print(f"\n  CER              : {cer_score:.4f}  ({cer_score*100:.1f}%)")

# --- Alignment visualization ---
# process_words aligns the sequences so visualize_alignment can render them
output = jiwer.process_words(reference, hypothesis)
print("\n=== Alignment visualization ===")
print(jiwer.visualize_alignment(output))

# --- Corpus-level vs. sentence-level WER ---
# Corpus-level WER pools all S+D+I across the entire corpus before dividing.
# Sentence-level average takes the mean of per-utterance WERs.
# They differ because short sentences with one error have very high per-sentence WER.
sentences_ref = [
    "hello world",              # 2 words
    "the quick brown fox jumps over the lazy dog",  # 9 words
    "good morning",             # 2 words
]
sentences_hyp = [
    "helo world",               # 1 substitution in 2 words -> 50% per-sentence
    "the quick brown cat jumps over the lazy dog",  # 1 sub in 9 words -> 11%
    "good morning",             # perfect -> 0%
]

corpus_wer = jiwer.wer(sentences_ref, sentences_hyp)
per_sentence_wers = [jiwer.wer(r, h) for r, h in zip(sentences_ref, sentences_hyp)]
sentence_avg_wer  = sum(per_sentence_wers) / len(per_sentence_wers)

print("=== Corpus-level vs. sentence-level WER ===")
for i, (r, h, w) in enumerate(zip(sentences_ref, sentences_hyp, per_sentence_wers)):
    print(f"  Sentence {i+1}: WER={w:.3f}  ref='{r}'  hyp='{h}'")
print(f"\n  Corpus-level WER     : {corpus_wer:.4f}")
print(f"  Sentence-average WER : {sentence_avg_wer:.4f}")
print()
print("The sentence with only 2 words inflates the sentence-average WER,")
print("while corpus-level WER correctly weights longer sentences more heavily.")

## Bootstrap Confidence Intervals for WER

A single WER number is just a point estimate. Bootstrapping gives you a 95% confidence interval: the range within which the true WER would fall 95% of the time if you re-ran the evaluation on a fresh sample of the same size. Wider intervals mean your test set is small and the estimate is noisy.

In [ ]:
import numpy as np
from scipy import stats
import jiwer

def bootstrap_confidence_interval(
    references: list,
    hypotheses: list,
    n_bootstrap: int = 1000,
    confidence: float = 0.95,
    seed: int = 42,
) -> dict:
    """
    Compute a bootstrap confidence interval for corpus-level WER.

    Resamples (references, hypotheses) pairs with replacement n_bootstrap times,
    computes WER for each resample, and returns the percentile CI.

    Args:
        references:  list of reference strings
        hypotheses:  list of hypothesis strings
        n_bootstrap: number of bootstrap resamples (1000 is standard)
        confidence:  confidence level, e.g. 0.95 for 95% CI
        seed:        random seed for reproducibility

    Returns:
        dict with keys: wer, ci_lower, ci_upper, ci_width, n_samples
    """
    assert len(references) == len(hypotheses), "references and hypotheses must be the same length"

    rng = np.random.default_rng(seed)
    n = len(references)

    # Point estimate
    point_wer = jiwer.wer(references, hypotheses)

    # Bootstrap resampling
    bootstrap_wers = []
    for _ in range(n_bootstrap):
        indices = rng.integers(0, n, size=n)
        resampled_refs = [references[i] for i in indices]
        resampled_hyps = [hypotheses[i] for i in indices]
        bootstrap_wers.append(jiwer.wer(resampled_refs, resampled_hyps))

    bootstrap_wers = np.array(bootstrap_wers)
    alpha = 1.0 - confidence
    ci_lower = np.percentile(bootstrap_wers, 100 * alpha / 2)
    ci_upper = np.percentile(bootstrap_wers, 100 * (1 - alpha / 2))

    return {
        "wer":       round(point_wer, 4),
        "ci_lower":  round(float(ci_lower), 4),
        "ci_upper":  round(float(ci_upper), 4),
        "ci_width":  round(float(ci_upper - ci_lower), 4),
        "n_samples": n,
    }


# --- Demo on synthetic ASR results ---
import random
random.seed(0)

# Build a 50-utterance synthetic test set with a mix of perfect and imperfect transcriptions
vocab = "the quick brown fox jumps over lazy dog cat sat on mat".split()
refs, hyps = [], []
for i in range(50):
    n_words = random.randint(3, 8)
    ref = " ".join(random.choices(vocab, k=n_words))
    # Introduce errors with ~20% probability per word
    hyp_words = []
    for w in ref.split():
        r = random.random()
        if r < 0.10:
            pass                            # deletion
        elif r < 0.20:
            hyp_words.append(random.choice(vocab))  # substitution
        else:
            hyp_words.append(w)             # correct
    hyps.append(" ".join(hyp_words))
    refs.append(ref)

result = bootstrap_confidence_interval(refs, hyps, n_bootstrap=2000)
print("Bootstrap WER Confidence Interval (95%)")
print(f"  WER       : {result['wer']:.4f}  ({result['wer']*100:.1f}%)")
print(f"  95% CI    : [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
print(f"  CI width  : {result['ci_width']:.4f}  ({result['ci_width']*100:.1f} pp)")
print(f"  n samples : {result['n_samples']}")
print()
print("How to read: we are 95% confident the true WER lies in that interval.")
print("A narrower interval means more test data -> more reliable estimate.")

## Batch Evaluation Loop

This section shows the full evaluation workflow: load a test set, run Whisper inference on all examples, collect predictions and references, then compute both corpus-level and sentence-level WER and explain why they differ.

In [ ]:
# pip install transformers torch soundfile  # uncomment if needed
import time
import numpy as np
import soundfile as sf
import torch
import jiwer
from transformers import pipeline as hf_pipeline

# --- Synthetic test set ---
# In real usage, replace with Common Voice or your own labeled audio files.
# Each entry: (audio_path, reference_transcript)
SYNTHETIC_TEST_DATA = [
    # We reuse the sine tones from earlier; Whisper will hallucinate text.
    # This illustrates the mechanics -- replace with real audio for real WER numbers.
    ("audio_samples/test_tone.wav",   ""),
    ("audio_samples/tone_220hz.wav",  ""),
    ("audio_samples/tone_880hz.wav",  ""),
]

# Create the audio files if they do not exist yet
import os, numpy as _np

os.makedirs("audio_samples", exist_ok=True)
for path, _ in SYNTHETIC_TEST_DATA:
    if not os.path.exists(path):
        freq = float(path.split("_")[1].replace("hz.wav", "")) if "hz" in path else 440.0
        t = _np.linspace(0, 2.0, 32000, endpoint=False)
        sf.write(path, (0.3 * _np.sin(2 * _np.pi * freq * t)).astype(_np.float32), 16000)

# --- Load Whisper pipeline ---
_dev = 0 if torch.cuda.is_available() else -1
eval_pipe = hf_pipeline("automatic-speech-recognition", model="openai/whisper-base", device=_dev)

# --- Run inference and collect results ---
all_refs, all_hyps, all_latencies = [], [], []

print("Running batch evaluation ...")
for audio_path, reference in SYNTHETIC_TEST_DATA:
    t0 = time.perf_counter()
    result = eval_pipe(audio_path, generate_kwargs={"language": "english"})
    latency = time.perf_counter() - t0

    hypothesis = result["text"].strip()
    all_refs.append(reference)
    all_hyps.append(hypothesis)
    all_latencies.append(latency)
    print(f"  {os.path.basename(audio_path):<28}  [{latency:.2f}s]  {repr(hypothesis[:50])}")

# --- Compute metrics ---
# Normalization transform
transform = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])

# Corpus-level WER: pools all S+D+I before dividing by total reference words
corpus_wer = jiwer.wer(all_refs, all_hyps,
                        reference_transform=transform,
                        hypothesis_transform=transform)

# Sentence-level WER average: mean of per-utterance WERs
per_sentence_wers = [
    jiwer.wer(r, h, reference_transform=transform, hypothesis_transform=transform)
    if r.strip() else float("nan")
    for r, h in zip(all_refs, all_hyps)
]
valid_wers = [w for w in per_sentence_wers if not (w != w)]  # filter NaN
sentence_avg_wer = sum(valid_wers) / len(valid_wers) if valid_wers else float("nan")

corpus_cer = jiwer.cer(all_refs, all_hyps) if any(r for r in all_refs) else float("nan")

print(f"\n{'='*50}")
print(f"  Corpus-level WER       : {corpus_wer:.4f}  ({corpus_wer*100:.1f}%)")
print(f"  Sentence-average WER   : {sentence_avg_wer:.4f}  ({sentence_avg_wer*100:.1f}%)")
print(f"  Corpus-level CER       : {corpus_cer:.4f}")
print(f"  Avg latency per file   : {sum(all_latencies)/len(all_latencies):.3f}s")
print(f"{'='*50}")
print()
print("Why corpus-level != sentence-average:")
print("  Corpus-level WER weights each utterance by its length (more words = more weight).")
print("  Sentence-average WER treats every utterance equally, so short utterances with")
print("  one error can dominate the average. Use corpus-level for overall model comparison.")

## Error Analysis: Best and Worst Utterances

Sorting utterances by per-sentence WER lets you quickly identify where the model struggles most. The 5 worst examples reveal systematic errors; the 5 best show what conditions the model handles well.

In [ ]:
import jiwer

# Synthetic labeled test set for demonstration.
# Replace with your actual (reference, hypothesis) pairs.
demo_refs = [
    "the cat sat on the mat",
    "she sells seashells by the seashore",
    "how much wood would a woodchuck chuck",
    "the quick brown fox jumps over the lazy dog",
    "to be or not to be that is the question",
    "all that glitters is not gold",
    "a stitch in time saves nine",
    "the early bird catches the worm",
    "actions speak louder than words",
    "better late than never",
]
demo_hyps = [
    "the cat sat on the mat",                           # perfect
    "she sells sea shells by the sea shore",            # 2 insertions
    "how much wood with a woodchuck chuck",             # 1 substitution
    "the quick brown fox jumped over the lazy dog",     # 1 substitution
    "to be or not to be that is the question",          # perfect
    "all that glitters is not gold",                    # perfect
    "a stitch in time saves nine people",               # 1 insertion
    "the early bird catches a worm",                    # 1 substitution
    "actions speak louder",                             # 3 deletions
    "better never",                                     # 2 deletions
]

# Compute per-utterance WER
_transform = jiwer.Compose([
    jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
    jiwer.Strip(), jiwer.ReduceToListOfListOfWords(),
])

utterance_wers = []
for ref, hyp in zip(demo_refs, demo_hyps):
    w = jiwer.wer(ref, hyp, reference_transform=_transform, hypothesis_transform=_transform)
    utterance_wers.append(w)

# Sort by WER descending
ranked = sorted(
    zip(utterance_wers, demo_refs, demo_hyps),
    key=lambda x: x[0],
    reverse=True,
)

TOP_N = 5

print(f"{'='*70}")
print(f"5 WORST utterances (highest WER)")
print(f"{'='*70}")
for wer_score, ref, hyp in ranked[:TOP_N]:
    print(f"  WER: {wer_score:.3f}")
    print(f"    REF: {ref}")
    print(f"    HYP: {hyp}")
    print()

print(f"{'='*70}")
print(f"5 BEST utterances (lowest WER)")
print(f"{'='*70}")
for wer_score, ref, hyp in ranked[-TOP_N:]:
    print(f"  WER: {wer_score:.3f}")
    print(f"    REF: {ref}")
    print(f"    HYP: {hyp}")
    print()

## Model Comparison Table: whisper-tiny vs. base vs. small

Running multiple Whisper sizes on the same test set and summarizing results in a pandas DataFrame gives a clean, comparable view of the accuracy-latency tradeoff.

In [ ]:
# pip install pandas transformers torch  # uncomment if needed
import time
import pandas as pd
import torch
import jiwer
from transformers import pipeline as hf_pipeline

# Reuse the demo test set from the error analysis cell above.
# demo_refs and demo_hyps are already defined.
# For a real evaluation: load audio files and transcribe with each model.
#
# Because we do not have real audio here, we simulate the comparison by
# adding calibrated noise to the ground-truth hypotheses to represent
# each model tier's expected error level.

MODELS = [
    ("openai/whisper-tiny",  "tiny"),
    ("openai/whisper-base",  "base"),
    ("openai/whisper-small", "small"),
]

_dev = 0 if torch.cuda.is_available() else -1
_jiwer_transform = jiwer.Compose([
    jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
    jiwer.Strip(), jiwer.ReduceToListOfListOfWords(),
])

# We will transcribe a single real audio file to measure latency.
# Replace AUDIO_FOR_LATENCY with a path to a real speech clip for meaningful latency numbers.
import os, numpy as _np, soundfile as _sf
_latency_path = "audio_samples/test_tone.wav"
if not os.path.exists(_latency_path):
    t = _np.linspace(0, 2.0, 32000, endpoint=False)
    _sf.write(_latency_path, (0.3 * _np.sin(2 * _np.pi * 440 * t)).astype(_np.float32), 16000)

comparison_rows = []
for model_id, label in MODELS:
    print(f"Evaluating {model_id} ...", end=" ", flush=True)
    pipe = hf_pipeline("automatic-speech-recognition", model=model_id, device=_dev)
    params_m = sum(p.numel() for p in pipe.model.parameters()) / 1e6

    # Latency: warm up then time one transcription
    _ = pipe(_latency_path)
    t0 = time.perf_counter()
    _ = pipe(_latency_path, generate_kwargs={"language": "english"})
    latency_s = time.perf_counter() - t0

    # For WER/CER: run inference on the demo sentences.
    # In a real setting, replace demo_refs/demo_hyps with actual model outputs.
    # Here we use the existing demo_hyps as a stand-in for the model outputs
    # (the WER numbers will be identical across models since we have no real audio,
    #  but the pipeline machinery is correct).
    pred_hyps = demo_hyps   # placeholder -- replace with pipe(audio_path) in real use

    model_wer = jiwer.wer(demo_refs, pred_hyps,
                           reference_transform=_jiwer_transform,
                           hypothesis_transform=_jiwer_transform)
    model_cer = jiwer.cer(demo_refs, pred_hyps)

    comparison_rows.append({
        "model":      label,
        "params (M)": round(params_m, 0),
        "WER":        round(model_wer, 4),
        "CER":        round(model_cer, 4),
        "latency (s)":round(latency_s, 3),
    })
    print(f"done  (WER={model_wer:.3f}, latency={latency_s:.2f}s)")
    del pipe

df = pd.DataFrame(comparison_rows)
print()
print(df.to_string(index=False))
print()
print("Note: WER/CER are identical here because we used the same synthetic hypotheses.")
print("Run on real labeled audio (e.g., Common Voice test split) for meaningful numbers.")

## Exercise: analyze_errors

Write a function that takes a list of predictions and references, extracts all substitution pairs (ref_word, hyp_word), and counts the most common ones. Also count how often a specific word (e.g., "the") is deleted entirely.

In [ ]:
from collections import Counter
import jiwer

def analyze_errors(predictions: list, references: list, top_n: int = 10) -> dict:
    """
    Analyze substitution and deletion patterns across a set of ASR outputs.

    Args:
        predictions: list of hypothesis strings
        references:  list of reference strings (same length as predictions)
        top_n:       how many top patterns to return

    Returns:
        dict with keys:
          "substitutions": Counter mapping (ref_word, hyp_word) -> count
          "deletions":     Counter mapping ref_word -> count
          "insertions":    Counter mapping hyp_word -> count
          "top_substitutions": top_n most common substitution pairs
          "top_deletions":     top_n most common deleted words
          "top_insertions":    top_n most common inserted words
    """
    # YOUR CODE HERE
    #
    # Hints:
    #   1. For each (ref, hyp) pair, call jiwer.process_words(ref, hyp).
    #      The returned object has an `.alignments` attribute: a list of
    #      AlignmentChunk objects, each with a `.type` ("equal", "substitute",
    #      "delete", "insert") and `.ref_start_idx`, `.ref_end_idx`,
    #      `.hyp_start_idx`, `.hyp_end_idx` fields.
    #
    #   2. Split ref and hyp into word lists. Use the alignment chunk indices
    #      to extract the actual words involved in each edit operation.
    #
    #   3. Accumulate counts in three Counter objects.
    #
    #   4. Return the dict described in the docstring.
    #
    # Verification (after implementing):
    #   test_refs = ["the cat sat on the mat", "the dog barked loudly"]
    #   test_hyps = ["a cat sat on a mat",     "the dog barked"]
    #   result = analyze_errors(test_hyps, test_refs)
    #   print(result["top_substitutions"])  # should show ('the','a') x2
    #   print(result["top_deletions"])      # should show 'loudly' x1

    raise NotImplementedError


# Quick smoke-test skeleton (uncomment after implementing)
# test_refs = ["the cat sat on the mat", "the dog barked loudly"]
# test_hyps = ["a cat sat on a mat",     "the dog barked"]
# result = analyze_errors(test_hyps, test_refs)
# print("Top substitutions:", result["top_substitutions"])
# print("Top deletions:    ", result["top_deletions"])